[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S15/2_Practica/en_clase/S15_01_seguimiento_experimentos_en_clase.ipynb)

# S15 · 01 — Seguimiento de experimentos y reproducibilidad · versión en clase

**Tiempo estimado:** 50 min en clase · **Sprint 15 (Proyecto: modelado y experimentos)** · píldora de apoyo

**Contexto.** En el proyecto final vuestro equipo probará muchos modelos y muchos ajustes. Al tercer día
nadie recuerda *"¿aquel 0,78 era con la semilla 1 o la 42? ¿con o sin la variable de antelación?"*. La
rúbrica (criterio C) premia el **registro de experimentos**, y un modelo que no se puede reproducir no se
puede defender. Aquí montamos un sistema de registro **muy simple** (un CSV) que funciona en cualquier
equipo. Usamos las reservas de TurisData como datos de práctica; en el proyecto usaréis los vuestros.

**Al terminar sabrás:**
1. Fijar semillas y anotar versiones para que otra persona obtenga *exactamente* tus resultados.
2. Registrar cada experimento (modelo, hiperparámetros, métrica, tiempo, datos) en un CSV compartido en Git.
3. Comparar modelos con **validación cruzada** y elegir el candidato **sin mirar el test**.
4. Detectar una **fuga de datos** por una métrica "demasiado buena".
5. (Opcional) Usar MLflow en local como alternativa más completa.

> 🏫 **Versión «en clase»** · ⏱ ≈ 50 min <!--reparto-->
>
> - Es **la parte de este cuaderno que se hace en el aula**, con el docente: ejercicios **1, 2, 3**.
> - Lo demás (ejercicios 🏠 y mini-reto) está en el [cuaderno completo](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S15/2_Practica/para_casa/S15_01_seguimiento_experimentos.ipynb), para hacer en casa.


In [ ]:
# Preparación (ejecuta esta celda siempre la primera)
import os, urllib.request

BASE_URL = "https://raw.githubusercontent.com/cCoolBoox/IFCD107-alumnado/main/datos/"

def dato(nombre):
    """Devuelve la ruta local de un archivo de datos del curso (lo descarga si no está)."""
    for carpeta in ("datos", "../datos", "../../datos", "../../../datos"):
        ruta = os.path.join(carpeta, nombre)
        if os.path.exists(ruta):
            return ruta
    os.makedirs("datos", exist_ok=True)
    ruta = os.path.join("datos", nombre)
    urllib.request.urlretrieve(BASE_URL + nombre, ruta)
    return ruta


In [ ]:
import os, sys, time, json, random, hashlib
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score

sns.set_theme(style="whitegrid")

## 1. Reproducibilidad: semillas y versiones

**Idea.** Muchos pasos usan azar (separar datos, inicializar pesos, submuestrear). Si no fijas la
**semilla**, cada ejecución da un resultado ligeramente distinto. Además, distintas **versiones** de las
librerías pueden cambiar resultados. Regla del equipo: **una función `fijar_semillas` al inicio de cada
notebook** y las versiones anotadas en el registro.

### Ejercicio 1 · 🏫 en clase
Escribe `fijar_semillas(semilla)` que fije la semilla de `random` y de `numpy` (`np.random.seed`),
guarde `PYTHONHASHSEED` en `os.environ` (como texto) y **devuelva** un generador
`np.random.default_rng(semilla)`. (Con TensorFlow añadirías `tf.random.set_seed(semilla)`.)

In [ ]:
def fijar_semillas(semilla):
    # TODO: Cada biblioteca con azar tiene su propia semilla; la variable de entorno debe ser un str; default_rng crea el generador moderno de numpy
    ...

SEMILLA = 42
rng = fijar_semillas(SEMILLA)
a = (random.random(), np.random.rand(), rng.random())
rng = fijar_semillas(SEMILLA)
b = (random.random(), np.random.rand(), rng.random())
assert a == b, "Al llamar dos veces a fijar_semillas(42) deberías obtener la misma secuencia aleatoria en random, np.random y el rng devuelto"
assert os.environ.get("PYTHONHASHSEED") == "42", "PYTHONHASHSEED debe guardarse como texto '42'"
print("Ejercicio 1 correcto ✔")

In [ ]:
# Ficha del entorno: cópiala en la memoria técnica y en el README del equipo
ENTORNO = {"python": sys.version.split()[0], "numpy": np.__version__, "pandas": pd.__version__, "scikit-learn": sklearn.__version__}
print(json.dumps(ENTORNO, indent=2))

## 2. El registro de experimentos: una fila por prueba

**Idea.** Cada vez que entrenas y evalúas, anotas una fila: **cuándo, qué modelo, con qué ajustes, con
qué datos, qué métrica y cuánto tardó**. Un CSV en el repositorio del equipo (`experimentos/registro.csv`)
se puede abrir con Excel, se versiona con Git y no requiere instalar nada.

Columnas mínimas: `id`, `fecha`, `responsable`, `modelo`, `hiperparametros` (JSON), `datos_hash`,
`semilla`, `metrica`, `valor_cv`, `desv_cv`, `tiempo_s`, `notas`.

### Ejercicio 2 · 🏫 en clase
Completa `registrar(...)`, que añade **una fila** al CSV `RUTA_REGISTRO`: si el archivo no existe, lo crea
con la cabecera; si existe, añade la fila sin repetir la cabecera. El `id` es el número de filas ya
existentes + 1. Devuelve el `id` asignado.

In [ ]:
RUTA_REGISTRO = "experimentos/registro.csv"
COLUMNAS = ["id", "fecha", "responsable", "modelo", "hiperparametros", "datos_hash", "semilla",
            "metrica", "valor_cv", "desv_cv", "tiempo_s", "notas"]
os.makedirs("experimentos", exist_ok=True)
if os.path.exists(RUTA_REGISTRO):
    os.remove(RUTA_REGISTRO)          # empezamos el laboratorio con el registro vacío

def registrar(responsable, modelo, hiperparametros, datos_hash, semilla, metrica, valor_cv, desv_cv, tiempo_s, notas=""):
    # TODO: Cuenta las filas con pd.read_csv si el archivo existe; crea un DataFrame de una fila y guárdalo con to_csv en modo "a" (header solo si es nuevo)
    ...

id1 = registrar("prueba", "ModeloDePrueba", {"a": 1}, "abc123", 42, "AUC", 0.5, 0.01, 0.1, "fila de prueba")
id2 = registrar("prueba", "ModeloDePrueba", {"a": 2}, "abc123", 42, "AUC", 0.6, 0.01, 0.1)
tabla_prueba = pd.read_csv(RUTA_REGISTRO)
assert (id1, id2) == (1, 2), "Los id deben ser 1, 2, ... (filas existentes + 1)"
assert list(tabla_prueba.columns) == COLUMNAS and len(tabla_prueba) == 2, "El CSV debe tener la cabecera una sola vez y 2 filas"
os.remove(RUTA_REGISTRO)              # borramos las filas de prueba
print("Ejercicio 2 correcto ✔")

## 3. Experimentos: baseline y modelos con validación cruzada

**Idea.** Para comparar modelos se usa **validación cruzada**: se parte el conjunto de entrenamiento en 5
trozos y se entrena 5 veces, evaluando cada vez en el trozo que no se ha visto. Obtenemos una **media**
y una **desviación** (¡importante para saber si una diferencia es real o ruido!). El **test** se
reserva para el final y se usa **una sola vez**. Con series temporales, la partición debe respetar el orden
del tiempo (`TimeSeriesSplit`), nunca al azar.

**Ejemplo resuelto:** preparamos datos y una función que entrena, evalúa y **registra automáticamente**.

In [ ]:
reservas = pd.read_csv(dato("reservas_turisdata.csv"), parse_dates=["fecha_reserva", "fecha_llegada"])
reservas["mes_llegada"] = reservas["fecha_llegada"].dt.month
X = reservas.drop(columns=["id_reserva", "fecha_reserva", "fecha_llegada", "cancelada"])
y = reservas["cancelada"]
X_ent, X_test, y_ent, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEMILLA)

# huella de los datos: si alguien cambia el CSV, el hash cambia y lo sabremos
DATOS_HASH = hashlib.md5(pd.util.hash_pandas_object(reservas, index=True).values.tobytes()).hexdigest()[:8]
print("Huella de los datos:", DATOS_HASH, "· entrenamiento:", X_ent.shape, "· test (sin tocar):", X_test.shape)

categoricas = X.select_dtypes("object").columns.tolist()
def crear_pipeline(estimador, columnas):
    numericas = [c for c in columnas if c not in categoricas]
    prep = ColumnTransformer([("num", StandardScaler(), numericas), ("cat", OneHotEncoder(handle_unknown="ignore"), categoricas)])
    return Pipeline([("prep", prep), ("m", estimador)])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEMILLA)

def experimento(nombre, estimador, hiperparametros, responsable="equipo", notas="", Xd=None):
    """Evalúa con CV (AUC), mide el tiempo y registra la fila. Devuelve (media, desviación)."""
    Xd = X_ent if Xd is None else Xd
    t0 = time.time()
    puntuaciones = cross_val_score(crear_pipeline(estimador, Xd.columns), Xd, y_ent, cv=cv, scoring="roc_auc")
    registrar(responsable, nombre, hiperparametros, DATOS_HASH, SEMILLA, "AUC", puntuaciones.mean(), puntuaciones.std(), time.time() - t0, notas)
    return puntuaciones.mean(), puntuaciones.std()

print("Baseline (predice la clase más frecuente):", experimento("Baseline", DummyClassifier(strategy="prior"), {"strategy": "prior"}, notas="referencia mínima"))
print("Regresión logística:", experimento("RegresionLogistica", LogisticRegression(max_iter=2000), {"C": 1.0}))

### Ejercicio 3 · 🏫 en clase
Registra dos experimentos más con la función `experimento(nombre, estimador, hiperparametros)`:
un **bosque aleatorio** `RandomForestClassifier(n_estimators=200, min_samples_leaf=5, random_state=SEMILLA, n_jobs=2)`
con nombre `"RandomForest"` y un **gradient boosting** `HistGradientBoostingClassifier(random_state=SEMILLA)` con
nombre `"HistGradientBoosting"` (hiperparámetros: los que uses, en un diccionario). Después lee el registro
en `tabla` (`pd.read_csv(RUTA_REGISTRO)`) y guarda en `candidato` el **nombre del modelo con mayor `valor_cv`**.

In [ ]:
# TODO: Llama a experimento dos veces; lee el CSV con RUTA_REGISTRO; idxmax sobre valor_cv te da la fila del mejor
...

assert len(tabla) == 4, "El registro debería tener 4 filas: baseline, logística, bosque y gradient boosting"
assert set(["RandomForest", "HistGradientBoosting"]) <= set(tabla["modelo"]), "Registra los dos modelos con los nombres indicados"
assert candidato == tabla.sort_values("valor_cv").iloc[-1]["modelo"], "candidato debe ser el modelo con mayor valor_cv"
print("Ejercicio 3 correcto ✔ — candidato provisional:", candidato)
tabla[["id", "modelo", "hiperparametros", "valor_cv", "desv_cv", "tiempo_s"]]

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
t = tabla.sort_values("valor_cv")
ax.barh(t["modelo"], t["valor_cv"], xerr=t["desv_cv"], color="tab:blue", capsize=4)
ax.axvline(0.5, color="gray", linestyle="--", label="Azar (AUC 0,5)")
ax.set_xlim(0.4, 0.85); ax.set_xlabel("AUC medio en validación cruzada (± desviación)")
ax.set_title("Comparativa de modelos (validación cruzada, sin usar el test)"); ax.legend(loc="lower right")
plt.tight_layout(); plt.show()

**Lectura importante:** si dos modelos difieren menos que su desviación, **no hay diferencia real**.
En ese caso elige el más simple, más rápido o más fácil de explicar. Anótalo en la columna `notas`.

---

## 🏁 Fin de la parte de clase

- Lo que queda (los ejercicios 🏠 y el mini-reto) está en el **[cuaderno completo](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S15/2_Practica/para_casa/S15_01_seguimiento_experimentos.ipynb)**.
- Ahí también tienes, en cada ejercicio de casa, su celda plegada «🆘 Solución de referencia» por si te quedas atascado.
